# Martin et al. (2023) acoustic aspiration-probe replication

Use a **T4 GPU** runtime. This evaluation only reads frozen acoustic checkpoints; it does not retrain or change Phase 1. Official MALD audio is about 1.5 GB and the formal nested-CV run is long. No Drive is used.

In [ ]:
# 1. Fetch the exact repository code without replacing Colab's PyTorch
import pathlib, subprocess, sys, importlib
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git', str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests>=2.31', 'scikit-learn>=1.3', 'matplotlib>=3.8'], check=True)
for name in list(sys.modules):
    if name == 'devlm' or name.startswith('devlm.'):
        del sys.modules[name]
sys.path.insert(0, str(PROJECT / 'src'))
importlib.invalidate_caches()
import devlm
print('devlm:', devlm.__file__)
print(subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Upload exactly the acoustic Phase-1 output ZIP (M01--M30)
from google.colab import files
import zipfile, shutil
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Upload exactly one acoustic-output ZIP')
archive = pathlib.Path(next(iter(uploaded)))
CHECKPOINT_ROOT = pathlib.Path('/content/acoustic_checkpoints')
if CHECKPOINT_ROOT.exists(): shutil.rmtree(CHECKPOINT_ROOT)
with zipfile.ZipFile(archive) as z:
    root = CHECKPOINT_ROOT.resolve()
    for info in z.infolist():
        if not (CHECKPOINT_ROOT / info.filename).resolve().is_relative_to(root): raise ValueError('Unsafe ZIP member')
    z.extractall(CHECKPOINT_ROOT)
from devlm.acoustic_aspiration_probe.run import discover_acoustic_checkpoints
checkpoints = discover_acoustic_checkpoints(CHECKPOINT_ROOT)
print('Validated', len(checkpoints), 'acoustic checkpoints:', [(i, round(h, 2)) for i, _, h in checkpoints])

In [ ]:
# 3. Formal run: verify paper-required Kaldi Fbank before any download/model work.
# Set a positive value ONLY for a labelled small smoke test; leave None for the paper-analog formal analysis.
SMOKE_MAX_ITEMS = None
from devlm.acoustic_aspiration_probe.run import _resolve_fbank_backend, run_replication
_, FBANK_BACKEND = _resolve_fbank_backend(allow_compatibility_fallback=SMOKE_MAX_ITEMS is not None)
print('Verified baseline backend:', FBANK_BACKEND)
OUTPUT = pathlib.Path('/content/acoustic_aspiration_replication_outputs')
MALD_CACHE = pathlib.Path('/content/mald_official_cache')
run_replication(CHECKPOINT_ROOT, OUTPUT, MALD_CACHE, max_items=SMOKE_MAX_ITEMS)
print('Complete:', OUTPUT)

In [ ]:
# 4. Download all results, plots, fold assignments, and manifest
import shutil
archive = shutil.make_archive('/content/acoustic_aspiration_replication_outputs', 'zip', OUTPUT)
files.download(archive)